# Отчёт по лабораторной работе №7

**Тема** — управление процессами.

**Студент** — ИУ5-35Б Штыка Денис Александрович.

**Вариант** — 23.

[Условие и генератор варианта](https://github.com/junaart/System_programming/blob/master/Lab_7/Lab_7.ipynb)


## Задание 1. Запуск программ через fork и execve

Файл `task1.asm` реализует командную строку для работ №5 и №6. Родитель считывает строку, формирует массив аргументов `argv` и вызывает `fork`. Дочерний процесс запускает нужный файл через `execve`.

| Команда | Действие |
| --- | --- |
| `task2 input.txt output.txt k` | Записать каждый k-й символ |
| `task5 input.txt output.txt k m` | Записать символы в раскачивающемся порядке |
| `task` | Запустить программу со спиралью из лабораторной №6 |
| `exit` | Дождаться активных программ и выйти |

Программы пятой лабораторной работают в фоне. Программа со спиралью получает терминал до выхода по клавише `l`, после чего родитель снова читает команды. Имена файлов с пробелами командная строка воспринимает как несколько аргументов.


## Задание 2. Обработка массива в потоках

Для варианта 23 генератор выдал **669 чисел**, размещение массива в куче и потоки через `clone`. В `task2.asm` память выделяется вызовом `brk`, числа получаются через `getrandom` и приводятся к диапазону от 1 до 10 000.

У каждого потока свой стек. Два потока сортируют отдельные копии массива для работы с порядковыми статистиками. Родитель ждёт через `futex`. При завершении потока ядро обнуляет его TID и будит ожидающий процесс.

| Операция | Как считается |
| --- | --- |
| Самая частая цифра | Частоты десяти цифр по всем числам. При равенстве выбирается меньшая цифра |
| Пятое число после минимума | Шестой элемент массива по возрастанию, с учётом повторов |
| Квантиль 0,75 | Элемент на позиции `ceil(0.75 × 669) = 502` при нумерации с единицы |
| Количество простых чисел | Проверка делителей от 2 до квадратного корня для каждого элемента |

Порядок вывода зависит от порядка завершения потоков.


## Сборка и запуск

Запустить `./run_fasm.sh` из корня проекта и в контейнере выполнить:

```bash
fasm Lab_5/task2.asm Lab_5/task2.o
ld -o Lab_5/task2 Lab_5/task2.o
fasm Lab_5/task5.asm Lab_5/task5.o
ld -o Lab_5/task5 Lab_5/task5.o

fasm Lab_6/task.asm Lab_6/task.o
gcc -no-pie Lab_6/task.o -l:libncurses.so.6 -l:libtinfo.so.6 -o Lab_6/task

fasm Lab_7/task1.asm Lab_7/task1.o
ld -o Lab_7/task1 Lab_7/task1.o
fasm Lab_7/task2.asm Lab_7/task2.o
ld -o Lab_7/task2 Lab_7/task2.o

./Lab_7/task1
./Lab_7/task2
```

Команда линковки шестой работы использует версии библиотек, доступные в образе проекта.


## Проверка

В контейнере Linux x86-64 команда `task2` при входе `abcdefghij` и `k = 3` записала `cfi`. Команда `task5` при входе `123456789`, `k = 5` и `m = 4` записала `564738291`. Проверена работа двух файловых команд одновременно и возврат из программы шестой работы по клавише `l`.

Для задания 2 массив из 669 чисел считан из памяти в GDB перед созданием потоков. Независимый расчёт на Python совпал с выводом программы: самая частая цифра — 7, пятое число после минимума — 131, квантиль 0,75 — 7561, простых чисел — 80. В каждом из 25 дополнительных запусков программа вывела четыре результата и завершилась.
